In [1]:
import pandas as pd

gyro = pd.read_csv('/content/drive/MyDrive/gyroformer/data/catalogs/gyronet_8k.csv')
tars = pd.read_csv('/content/drive/MyDrive/gyroformer/data/catalogs/tars_table_2.csv')

print("=== gyronet_8k ===")
print(f"Shape: {gyro.shape}")
print(gyro.columns.tolist())

print("\n=== TARS ===")
print(f"Shape: {tars.shape}")
print(tars.columns.tolist())

=== gyronet_8k ===
Shape: (8349, 23)
['GaiaDR3_ID', 'cluster', 'subgroup', 'fiducial_age_Myr', 'Prot', 'BPRP_0', 'G_0', 'e_BPRP_0', 'dr3_parallax_zpt_corrected', 'dr3_ruwe', 'mem_prob_hdbscan', 'cmd_exclude', 'dered_exclude', 'phot_exclude', 'phot_bp_rp_excess_factor', 'astrometric_excess_noise_sig', 'phot_bp_rp_excess_factor_t', 'astrometric_excess_noise_sig_t', 'noise_sig_detected', 'spec_coverage', 'e_Prot_true', 'e_Prot_imputed', 'mem_prob_source']

=== TARS ===
Shape: (944056, 36)
['TICID', 'dr2_source_id', 'dr3_source_id', 'ra', 'dec', 'pmra', 'pmdec', 'parallax', 'teff', 'Tmag', 'phot_g_mean_mag', 'phot_g_mean_mag_0', 'phot_bp_mean_mag', 'phot_bp_mean_mag_0', 'phot_rp_mean_mag', 'phot_rp_mean_mag_0', 'BpmRp0', 'extinction_a0', 'ruwe', 'non_single_star', 'adopted_period', 'adopted_period_unc', 'flag_multiple_periods', 'flag_possible_binary', 'final_n_contams', 'flag_doubled_period', 'n_secs', 'n_sec_ratio', 'median_amplitude', 'sectors', 'sector_periods', 'sector_sys_probs', 'sec

In [2]:
print("=== Join key inspection ===")
print(f"gyro GaiaDR3_ID dtype: {gyro['GaiaDR3_ID'].dtype}")
print(f"tars dr3_source_id dtype: {tars['dr3_source_id'].dtype}")

print(f"\ngyro sample IDs:\n{gyro['GaiaDR3_ID'].head().tolist()}")
print(f"\ntars sample IDs:\n{tars['dr3_source_id'].head().tolist()}")

print(f"\ngyro nulls in GaiaDR3_ID: {gyro['GaiaDR3_ID'].isna().sum()}")
print(f"tars nulls in dr3_source_id: {tars['dr3_source_id'].isna().sum()}")

=== Join key inspection ===
gyro GaiaDR3_ID dtype: int64
tars dr3_source_id dtype: int64

gyro sample IDs:
[5290646440127895936, 5290651323511794816, 5290652663541576832, 5290653350736340736, 5290654205428930688]

tars sample IDs:
[6220284483485052800, 6220293971070316928, 6221815798241950976, 6221815798241951872, 6222011511311483392]

gyro nulls in GaiaDR3_ID: 0
tars nulls in dr3_source_id: 0


In [3]:
overlap = gyro['GaiaDR3_ID'].isin(tars['dr3_source_id'])

print(f"Labeled stars in TARS:     {overlap.sum()} / {len(gyro)}")
print(f"Labeled stars NOT in TARS: {(~overlap).sum()} / {len(gyro)}")
print(f"Coverage: {overlap.mean()*100:.1f}%")

Labeled stars in TARS:     2823 / 8349
Labeled stars NOT in TARS: 5526 / 8349
Coverage: 33.8%


In [4]:
coverage_by_cluster = gyro.groupby('cluster').apply(
    lambda df: pd.Series({
        'total': len(df),
        'in_tars': df['GaiaDR3_ID'].isin(tars['dr3_source_id']).sum()
    })
).reset_index()

coverage_by_cluster['coverage_pct'] = (coverage_by_cluster['in_tars'] / coverage_by_cluster['total'] * 100).round(1)
coverage_by_cluster = coverage_by_cluster.sort_values('coverage_pct', ascending=False)

print(coverage_by_cluster.to_string(index=False))

     cluster  total  in_tars  coverage_pct
        APer    238      219          92.0
      IC2602    193      170          88.1
Collinder135    249      216          86.7
    NGC2451A    149      128          85.9
      IC2391    113       96          85.0
      Hyades    178      133          74.7
      Taurus     93       63          67.7
    Pleiades    855      531          62.1
     NGC3532    225      119          52.9
     NGC2547    195       97          49.7
      ScoCen   1041      512          49.2
     NGC2516    575      227          39.5
      NGC752      8        3          37.5
    Praesepe    995      291          29.2
         M34     83       10          12.0
 Ruprecht147     42        4           9.5
     NGC2281    128        4           3.1
         M67    283        0           0.0
         M50    657        0           0.0
         M37    348        0           0.0
         M35    537        0           0.0
        HPer    420        0           0.0
     NGC175

/tmp/ipykernel_3266/2970776069.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  coverage_by_cluster = gyro.groupby('cluster').apply(


In [5]:
age_coverage = gyro.groupby('fiducial_age_Myr').apply(
    lambda df: pd.Series({
        'total': len(df),
        'in_tars': df['GaiaDR3_ID'].isin(tars['dr3_source_id']).sum()
    }), include_groups=False
).reset_index()

age_coverage['coverage_pct'] = (age_coverage['in_tars'] / age_coverage['total'] * 100).round(1)
age_coverage = age_coverage.sort_values('fiducial_age_Myr')

print(age_coverage.to_string(index=False))

 fiducial_age_Myr  total  in_tars  coverage_pct
             1.57     16       10          62.5
             1.73     18       14          77.8
             2.01     16        9          56.2
             2.49     10        5          50.0
             2.59     12        6          50.0
             3.27      9        8          88.9
             3.40      4        4         100.0
             3.70    163       66          40.5
             6.00     73       37          50.7
             6.22      7        6          85.7
             7.85    103       45          43.7
            10.15    289      151          52.2
            10.25    140       76          54.3
            12.65    143       71          49.7
            13.70     57       29          50.9
            14.35      2        1          50.0
            15.00    420        0           0.0
            17.66      1        1         100.0
            18.75     71       36          50.7
            26.00    249      216       

In [6]:
not_in_tars = gyro[~gyro['GaiaDR3_ID'].isin(tars['dr3_source_id'])]

print(f"Stars not in TARS: {len(not_in_tars)}")
print(f"\nProt null:     {not_in_tars['Prot'].isna().sum()}")
print(f"Prot non-null: {not_in_tars['Prot'].notna().sum()}")
print(f"\nProt sample (non-null):\n{not_in_tars['Prot'].dropna().describe()}")

Stars not in TARS: 5526

Prot null:     0
Prot non-null: 5526

Prot sample (non-null):
count    5526.000000
mean        7.398995
std        13.169255
min         0.031000
25%         0.849000
50%         3.538500
75%         9.233500
max       172.000000
Name: Prot, dtype: float64


In [7]:
print("=== Quality flag counts (1 = excluded) ===")
for col in ['cmd_exclude', 'dered_exclude', 'phot_exclude', 'noise_sig_detected']:
    vc = gyro[col].value_counts(dropna=False)
    print(f"\n{col}:\n{vc}")

=== Quality flag counts (1 = excluded) ===

cmd_exclude:
cmd_exclude
0    8349
Name: count, dtype: int64

dered_exclude:
dered_exclude
0    8349
Name: count, dtype: int64

phot_exclude:
phot_exclude
0    8349
Name: count, dtype: int64

noise_sig_detected:
noise_sig_detected
1    5403
0    2946
Name: count, dtype: int64


In [8]:
print("=== TARS quality flags ===")
for col in ['flag_possible_binary', 'flag_multiple_periods', 'flag_doubled_period']:
    vc = tars[col].value_counts(dropna=False)
    print(f"\n{col}:\n{vc}")

=== TARS quality flags ===

flag_possible_binary:
flag_possible_binary
False    776639
True     167417
Name: count, dtype: int64

flag_multiple_periods:
flag_multiple_periods
False    853195
True      90861
Name: count, dtype: int64

flag_doubled_period:
flag_doubled_period
True     524963
False    419093
Name: count, dtype: int64


In [9]:
print("=== Reliability-related columns ===")
for col in ['sector_sig_probs', 'sector_sys_probs', 'sector_match_probs', 'sector_alias_probs']:
    print(f"\n{col} sample:\n{tars[col].head(10).tolist()}")

=== Reliability-related columns ===

sector_sig_probs sample:
['0.99', '0.98', '0.99,0.99', '1.00,0.99', '1.00,1.00', '1.00,1.00', '1.00,1.00,1.00', '1.00,1.00', '0.94', '0.99,1.00']

sector_sys_probs sample:
['0.01', '0.02', '0.01,0.01', '0.00,0.01', '0.00,0.00', '0.00,0.00', '0.00,0.00,0.00', '0.00,0.00', '0.06', '0.01,0.00']

sector_match_probs sample:
['0.00', '0.84', '0.92,0.98', '0.99,0.99', '0.97,0.97', '0.98,0.93', '1.00,0.91,1.00', '1.00,0.99', '0.17', '0.88,0.88']

sector_alias_probs sample:
['1.00', '0.15', '0.09,0.02', '0.01,0.01', '0.03,0.03', '0.02,0.07', '0.00,0.09,0.00', '0.00,0.01', '0.83', '0.12,0.12']


In [10]:
cut1 = tars['flag_possible_binary'] == False
cut2 = tars['flag_multiple_periods'] == False
cut3 = tars['final_n_contams'] == 0
cut4 = tars['n_secs'] > 1

print(f"Total TARS:                          {len(tars):>7}")
print(f"After flag_possible_binary==False:   {cut1.sum():>7}")
print(f"After + flag_multiple_periods==False:{(cut1 & cut2).sum():>7}")
print(f"After + final_n_contams==0:          {(cut1 & cut2 & cut3).sum():>7}")
print(f"After + n_secs > 1:                  {(cut1 & cut2 & cut3 & cut4).sum():>7}")

Total TARS:                           944056
After flag_possible_binary==False:    776639
After + flag_multiple_periods==False: 705535
After + final_n_contams==0:           313916
After + n_secs > 1:                   202223


In [11]:
labeled_in_tars = tars[tars['dr3_source_id'].isin(gyro['GaiaDR3_ID'])]

print(f"Labeled stars in TARS: {len(labeled_in_tars)}")
print(f"After flag_possible_binary==False:   {(labeled_in_tars['flag_possible_binary']==False).sum()}")
print(f"After + flag_multiple_periods==False: {((labeled_in_tars['flag_possible_binary']==False) & (labeled_in_tars['flag_multiple_periods']==False)).sum()}")
print(f"After + final_n_contams==0:           {((labeled_in_tars['flag_possible_binary']==False) & (labeled_in_tars['flag_multiple_periods']==False) & (labeled_in_tars['final_n_contams']==0)).sum()}")
print(f"After + n_secs > 1:                   {((labeled_in_tars['flag_possible_binary']==False) & (labeled_in_tars['flag_multiple_periods']==False) & (labeled_in_tars['final_n_contams']==0) & (labeled_in_tars['n_secs']>1)).sum()}")

Labeled stars in TARS: 2823
After flag_possible_binary==False:   2496
After + flag_multiple_periods==False: 2355
After + final_n_contams==0:           1459
After + n_secs > 1:                   1070


In [12]:
not_in_tars = gyro[~gyro['GaiaDR3_ID'].isin(tars['dr3_source_id'])]

kepler_clusters = ['NGC6811', 'NGC6819', 'NGC6866', 'Praesepe', 'M67']
k2_clusters = ['Pleiades', 'Hyades', 'Praesepe', 'M35', 'M37', 'M44', 'M50',
                'NGC752', 'NGC1647', 'NGC1750', 'NGC1758', 'NGC1817', 'NGC2281',
                'NGC2516', 'NGC2547', 'NGC2548', 'NGC3532', 'Ruprecht147',
                'NGC6819', 'HPer', 'M34']

not_in_tars['likely_kepler'] = not_in_tars['cluster'].isin(kepler_clusters)
not_in_tars['likely_k2'] = not_in_tars['cluster'].isin(k2_clusters)

print(f"Not in TARS: {len(not_in_tars)}")
print(f"Likely Kepler: {not_in_tars['likely_kepler'].sum()}")
print(f"Likely K2:     {not_in_tars['likely_k2'].sum()}")
print(f"Neither:       {(~not_in_tars['likely_kepler'] & ~not_in_tars['likely_k2']).sum()}")

print(f"\nClusters with no TARS and not in either list:")
neither = not_in_tars[~not_in_tars['likely_kepler'] & ~not_in_tars['likely_k2']]
print(neither['cluster'].value_counts())

Not in TARS: 5526
Likely Kepler: 1519
Likely K2:     4171
Neither:       672

Clusters with no TARS and not in either list:
cluster
ScoCen          529
Collinder135     33
Taurus           30
IC2602           23
NGC2451A         21
APer             19
IC2391           17
Name: count, dtype: int64


/tmp/ipykernel_3266/4002339610.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_in_tars['likely_kepler'] = not_in_tars['cluster'].isin(kepler_clusters)
/tmp/ipykernel_3266/4002339610.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_in_tars['likely_k2'] = not_in_tars['cluster'].isin(k2_clusters)


In [13]:
print("=== Labeled Set Coverage Summary ===")
print(f"Total labeled stars:              {len(gyro):>6}")
print(f"  In TARS (light curve on MAST):  {gyro['GaiaDR3_ID'].isin(tars['dr3_source_id']).sum():>6}  ({33.8}%)")
print(f"  Not in TARS:                    {(~gyro['GaiaDR3_ID'].isin(tars['dr3_source_id'])).sum():>6}  ({66.2}%)")
print(f"    Likely Kepler:                {1519:>6}")
print(f"    Likely K2:                    {4171:>6}")
print(f"    Neither (ScoCen etc):         {672:>6}")

print(f"\n=== TARS Unlabeled Corpus ===")
print(f"Total TARS:                       {len(tars):>6}")
print(f"Strict quality cut (paper §4.4):  {202223:>6}  (binary+multiple+contam+multisec)")
print(f"Binary flag only:                 {(tars['flag_possible_binary']==False).sum():>6}")

print(f"\n=== Age Coverage Gap ===")
print(f"Clusters with zero TARS coverage: M67 (3.87Gyr), NGC6819 (2.31Gyr),")
print(f"  M50, M35, M37, HPer, NGC6811, NGC6866 — all need K2/Kepler")

=== Labeled Set Coverage Summary ===
Total labeled stars:                8349
  In TARS (light curve on MAST):    2823  (33.8%)
  Not in TARS:                      5526  (66.2%)
    Likely Kepler:                  1519
    Likely K2:                      4171
    Neither (ScoCen etc):            672

=== TARS Unlabeled Corpus ===
Total TARS:                       944056
Strict quality cut (paper §4.4):  202223  (binary+multiple+contam+multisec)
Binary flag only:                 776639

=== Age Coverage Gap ===
Clusters with zero TARS coverage: M67 (3.87Gyr), NGC6819 (2.31Gyr),
  M50, M35, M37, HPer, NGC6811, NGC6866 — all need K2/Kepler


In [15]:
from astroquery.mast import Catalogs
from astroquery.vizier import Vizier
import warnings
warnings.filterwarnings('ignore')

# Check if there are published Gaia-Kepler and Gaia-K2 crossmatch tables on VizieR
v = Vizier(row_limit=5)

# Berger+2018 is the standard Gaia-Kepler crossmatch
result_kepler = v.find_catalogs('Gaia Kepler crossmatch Berger')
for k, v_ in list(result_kepler.items())[:5]:
    print(k, '—', v_.description)

print()

# Hardegree-Ullman is standard for Gaia-K2
result_k2 = v.find_catalogs('Gaia K2 crossmatch')
for k, v_ in list(result_k2.items())[:5]:
    print(k, '—', v_.description)

I/324 — The Initial Gaia Source List (IGSL) (Smart, 2013)
I/337 — Gaia DR1 (Gaia Collaboration, 2016)
I/345 — Gaia DR2 (Gaia Collaboration, 2018)
I/347 — Distances to 1.33 billion stars in Gaia DR2 (Bailer-Jones+, 2018)
I/350 — Gaia EDR3 (Gaia Collaboration, 2020)

I/324 — The Initial Gaia Source List (IGSL) (Smart, 2013)
I/337 — Gaia DR1 (Gaia Collaboration, 2016)
I/345 — Gaia DR2 (Gaia Collaboration, 2018)
I/347 — Distances to 1.33 billion stars in Gaia DR2 (Bailer-Jones+, 2018)
I/350 — Gaia EDR3 (Gaia Collaboration, 2020)


In [16]:
v = Vizier(row_limit=5)

print("=== Berger+2018 Gaia-Kepler (J/ApJ/866/99) ===")
try:
    r = v.get_catalogs('J/ApJ/866/99')
    print(f"Tables: {[t.meta.get('name', i) for i, t in enumerate(r)]}")
    print(r[0].colnames)
except Exception as e:
    print(f"Error: {e}")

print("\n=== Hardegree-Ullman+2022 Gaia-K2 (J/ApJS/259/25) ===")
try:
    r2 = v.get_catalogs('J/ApJS/259/25')
    print(f"Tables: {[t.meta.get('name', i) for i, t in enumerate(r2)]}")
    print(r2[0].colnames)
except Exception as e:
    print(f"Error: {e}")

=== Berger+2018 Gaia-Kepler (J/ApJ/866/99) ===
Tables: ['J/ApJ/866/99/table1', 'J/ApJ/866/99/table2']
['Np', 'KIC', 'Gaia', 'Teff', 'e_Teff', 'Dist', 'E_Dist', 'e_Dist', 'R*', 'E_R*', 'e_R*', 'Av', 'Evol', 'Bin', 'M17', 'Simbad', '_RA', '_DE']

=== Hardegree-Ullman+2022 Gaia-K2 (J/ApJS/259/25) ===
Error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [17]:
v_full = Vizier(row_limit=-1)

print("Fetching Berger+2018 Gaia-Kepler crossmatch...")
berger = v_full.get_catalogs('J/ApJ/866/99')[0]
berger_df = berger.to_pandas()

print(f"Berger catalog size: {len(berger_df)}")
print(f"Gaia column sample: {berger_df['Gaia'].head().tolist()}")
print(f"Gaia dtype: {berger_df['Gaia'].dtype}")

Fetching Berger+2018 Gaia-Kepler crossmatch...
Berger catalog size: 177911
Gaia column sample: [2050233807328471424, 2050233601176543104, 2050230543159814656, 2050230611879323904, 2050231848829944320]
Gaia dtype: int64


In [18]:
# Naive match — will undercount due to DR2 vs DR3 ID difference
naive_match = gyro['GaiaDR3_ID'].isin(berger_df['Gaia'])
print(f"Naive DR3-vs-DR2 match: {naive_match.sum()} / {len(gyro)}")

# TARS has both dr2 and dr3 IDs — we can use it as a bridge
# Check if TARS has dr2 IDs for our labeled stars
labeled_in_tars_full = tars[tars['dr3_source_id'].isin(gyro['GaiaDR3_ID'])][['dr2_source_id', 'dr3_source_id']]
print(f"\nTARS has DR2 IDs for {len(labeled_in_tars_full)} of our labeled stars")
print(f"DR2 nulls in those rows: {labeled_in_tars_full['dr2_source_id'].isna().sum()}")

# Now try bridging through TARS for the Kepler clusters specifically
kepler_stars = gyro[gyro['cluster'].isin(kepler_clusters)]
kepler_with_dr2 = kepler_stars.merge(
    labeled_in_tars_full, left_on='GaiaDR3_ID', right_on='dr3_source_id', how='left'
)
print(f"\nKepler cluster stars: {len(kepler_stars)}")
print(f"Of those, have DR2 via TARS bridge: {kepler_with_dr2['dr2_source_id'].notna().sum()}")

Naive DR3-vs-DR2 match: 398 / 8349

TARS has DR2 IDs for 2823 of our labeled stars
DR2 nulls in those rows: 0

Kepler cluster stars: 1810
Of those, have DR2 via TARS bridge: 291


In [21]:
query_cols = """
SELECT column_name
FROM tap_schema.columns
WHERE table_name = 'gaiadr3.dr2_neighbourhood'
"""

job = Gaia.launch_job(query_cols)
print(job.get_results())

       column_name       
-------------------------
            dr2_source_id
            dr3_source_id
         angular_distance
     magnitude_difference
proper_motion_propagation


In [22]:
from astroquery.gaia import Gaia

# Get DR2 IDs for all labeled stars not bridged via TARS
# First build the full set of DR2 IDs we need
# Use TARS bridge where available, query Gaia archive for the rest

kepler_star_dr3_ids = gyro[gyro['cluster'].isin(kepler_clusters)]['GaiaDR3_ID'].tolist()

# Query Gaia DR3 archive for DR2 equivalents
id_list = ','.join(str(i) for i in kepler_star_dr3_ids)

query = f"""
SELECT dr3_source_id, dr2_source_id
FROM gaiadr3.dr2_neighbourhood
WHERE dr3_source_id IN ({id_list})
AND angular_distance < 0.5
AND magnitude_difference < 0.5
"""

print("Querying Gaia archive for DR2->DR3 crossmatch...")
job = Gaia.launch_job(query)
result = job.get_results().to_pandas()
print(f"Results: {len(result)}")
print(result.head())

Querying Gaia archive for DR2->DR3 crossmatch...
Results: 1735
        dr3_source_id       dr2_source_id
0  659440138937561344  659440138937561344
1  664477968073854720  664477968073854720
2  604910924910708224  604910924910708224
3  664508616960346752  664508616960346752
4  661026738512909440  661026738512909440


In [23]:
# Merge DR3->DR2 bridge with Berger catalog
berger_match = result.merge(berger_df, left_on='dr2_source_id', right_on='Gaia', how='inner')

print(f"Kepler cluster stars with DR3->DR2 bridge: {len(result)}")
print(f"Of those, found in Berger+2018 (have KIC): {len(berger_match)}")
print(f"Unique DR3 IDs matched: {berger_match['dr3_source_id'].nunique()}")
print(f"\nSample KIC IDs: {berger_match['KIC'].head().tolist()}")

Kepler cluster stars with DR3->DR2 bridge: 1735
Of those, found in Berger+2018 (have KIC): 402
Unique DR3 IDs matched: 401

Sample KIC IDs: [9838496, 9656537, 9836149, 5112670, 9716107]


In [24]:
# Add DR2 and KIC info back to the gyro labeled set for Kepler stars
kepler_stars_df = gyro[gyro['cluster'].isin(kepler_clusters)].copy()
kepler_stars_df = kepler_stars_df.merge(result, left_on='GaiaDR3_ID', right_on='dr3_source_id', how='left')
kepler_stars_df = kepler_stars_df.merge(berger_df[['Gaia','KIC']], left_on='dr2_source_id', right_on='Gaia', how='left')

print(kepler_stars_df.groupby('cluster').apply(
    lambda df: pd.Series({
        'total': len(df),
        'has_dr2': df['dr2_source_id'].notna().sum(),
        'has_kic': df['KIC'].notna().sum()
    }), include_groups=False
).reset_index().to_string(index=False))

 cluster  total  has_dr2  has_kic
     M67    283      277        0
 NGC6811    279      277      223
 NGC6819    132      132       70
 NGC6866    125      125      113
Praesepe    995      928        0


In [26]:
print("Fetching Hardegree-Ullman+2022 Gaia-K2 crossmatch (J/ApJS/247/28...")
try:
    r_k2 = v_full.get_catalogs('J/ApJS/247/28')
    print(f"Number of tables: {len(r_k2)}")
    for i, t in enumerate(r_k2):
        print(f"  Table {i}: {t.meta.get('name', '?')} — {len(t)} rows — cols: {t.colnames[:8]}")
except Exception as e:
    print(f"Error: {e}")

Fetching Hardegree-Ullman+2022 Gaia-K2 crossmatch (J/ApJS/247/28...
Number of tables: 2
  Table 0: J/ApJS/247/28/table1 — 244337 rows — cols: ['Np', 'EPIC', 'Camp', 'PS', 'Gaia', 'LAMOST', 'gmag', 'rmag']
  Table 1: J/ApJS/247/28/table3 — 816 rows — cols: ['EPIC', 'ID', 'Planet', 'Rp/R*', 'Per', 'r_Per', 'SpT', 'Teff']


In [28]:
query_k2_full = f"""
SELECT dr3_source_id, dr2_source_id
FROM gaiadr3.dr2_neighbourhood
WHERE dr3_source_id IN ({id_list_k2})
AND angular_distance < 0.5
AND magnitude_difference < 0.5
"""

print("Querying Gaia archive (async, no row limit)...")
job_k2_full = Gaia.launch_job_async(query_k2_full)
result_k2_full = job_k2_full.get_results().to_pandas()
print(f"DR3->DR2 bridge results: {len(result_k2_full)} / {len(k2_stars_df)}")

Querying Gaia archive (async, no row limit)...


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
DR3->DR2 bridge results: 5380 / 6280


In [29]:
k2_stars_df2 = gyro[gyro['cluster'].isin(all_k2_clusters)].copy()
k2_stars_df2 = k2_stars_df2.merge(result_k2_full, left_on='GaiaDR3_ID', right_on='dr3_source_id', how='left')
k2_stars_df2 = k2_stars_df2.merge(k2_df[['Gaia','EPIC']], left_on='dr2_source_id', right_on='Gaia', how='left')

print(f"K2 cluster stars with EPIC ID: {k2_stars_df2['EPIC'].notna().sum()} / {len(k2_stars_df2)}")
print(k2_stars_df2.groupby('cluster').apply(
    lambda df: pd.Series({
        'total': len(df),
        'has_epic': df['EPIC'].notna().sum()
    }), include_groups=False
).reset_index().to_string(index=False))

K2 cluster stars with EPIC ID: 2761 / 6936
    cluster  total  has_epic
       HPer    420         0
     Hyades    179        73
        M34     83         0
        M35    537         0
        M37    348         0
        M50    657         0
        M67    448       276
    NGC1647     35        28
    NGC1750     39        27
    NGC1758     21        17
    NGC1817     57        53
    NGC2281    128         0
    NGC2516    575         0
    NGC2547    195         0
    NGC2548     62         0
    NGC3532    225         0
    NGC6819    132         0
     NGC752      8         0
   Pleiades    855       578
   Praesepe   1890      1675
Ruprecht147     42        34


In [30]:
in_tars = gyro['GaiaDR3_ID'].isin(tars['dr3_source_id'])

kepler_matched = set(kepler_stars_df[kepler_stars_df['KIC'].notna()]['GaiaDR3_ID'])
k2_matched = set(k2_stars_df2[k2_stars_df2['EPIC'].notna()]['GaiaDR3_ID'])

gyro['source'] = 'none'
gyro.loc[in_tars, 'source'] = 'TARS'
gyro.loc[gyro['GaiaDR3_ID'].isin(kepler_matched), 'source'] = 'Kepler'
gyro.loc[gyro['GaiaDR3_ID'].isin(k2_matched) & ~in_tars, 'source'] = 'K2'

print("=== Overall labeled set coverage ===")
print(gyro['source'].value_counts())
print(f"\nTotal recoverable: {(gyro['source'] != 'none').sum()} / {len(gyro)} ({(gyro['source'] != 'none').mean()*100:.1f}%)")
print(f"No light curve found: {(gyro['source'] == 'none').sum()}")

=== Overall labeled set coverage ===
source
none      4156
TARS      2823
K2         968
Kepler     402
Name: count, dtype: int64

Total recoverable: 4193 / 8349 (50.2%)
No light curve found: 4156


In [31]:
print(gyro.groupby('cluster').apply(
    lambda df: pd.Series({
        'total': len(df),
        'TARS': (df['source']=='TARS').sum(),
        'Kepler': (df['source']=='Kepler').sum(),
        'K2': (df['source']=='K2').sum(),
        'none': (df['source']=='none').sum(),
    }), include_groups=False
).reset_index().sort_values('none', ascending=False).to_string(index=False))

     cluster  total  TARS  Kepler  K2  none
         M50    657     0       0   0   657
         M35    537     0       0   0   537
      ScoCen   1041   512       0   0   529
        HPer    420     0       0   0   420
     NGC2516    575   227       0   0   348
         M37    348     0       0   0   348
    Praesepe    995   291       0 512   192
         M67    283     0       0 111   172
    Pleiades    855   531       0 187   137
     NGC2281    128     4       0   0   124
     NGC3532    225   119       0   0   106
     NGC2547    195    97       0   0    98
         M34     83    10       0   0    73
     NGC2548     62     0       0   0    62
     NGC6819    132     0      70   0    62
     NGC6811    277     0     221   0    56
      Hyades    178   133       0   4    41
Collinder135    249   216       0   0    33
      Taurus     93    63       0   0    30
      IC2602    193   170       0   0    23
    NGC2451A    149   128       0   0    21
        APer    238   219       

In [32]:
from astroquery.mast import Observations

# Take a sample of unrecovered stars across different clusters
unrecovered = gyro[gyro['source'] == 'none'].copy()

# Sample 20 across different clusters
sample = unrecovered.groupby('cluster').first().reset_index().head(20)
print(f"Testing {len(sample)} unrecovered stars across clusters")
print(sample[['GaiaDR3_ID', 'cluster', 'Prot']].to_string(index=False))

Testing 20 unrecovered stars across clusters
         GaiaDR3_ID      cluster      Prot
 251387120195665152         APer  0.337000
5589411172768195840 Collinder135  0.470000
 458360490378051456         HPer  8.060000
 144506309274714624       Hyades  0.150000
5317887321743547264       IC2391  0.770000
5239799391792479488       IC2602  0.330000
 337180328879557120          M34  0.940000
3425496835312410112          M35  0.942000
3450988286555378176          M37 16.989132
3050039590390925056          M50  1.245000
 598883642685066368          M67 48.700000
3409891359404877696      NGC1647  1.250000
3418675155938540672      NGC1750 16.813000
3418695630046147840      NGC1758  0.671000
3393981052490356992      NGC1817  9.596000
 945468231955682560      NGC2281  4.320000
5538729734043145600     NGC2451A  3.360000
5290646440127895936      NGC2516  1.897000
5514355176164451712      NGC2547  0.294000
3064466866575475712      NGC2548  1.670000


In [33]:
from astroquery.mast import Observations
import time

results = []
for _, row in sample.iterrows():
    try:
        obs = Observations.query_criteria(
            objectname=f"Gaia DR3 {row['GaiaDR3_ID']}",
            obs_collection=['Kepler', 'K2', 'TESS'],
            dataproduct_type='timeseries'
        )
        n = len(obs)
        missions = list(set(obs['obs_collection'].tolist())) if n > 0 else []
    except Exception as e:
        n = 0
        missions = [f'error: {e}']
    results.append({'GaiaDR3_ID': row['GaiaDR3_ID'], 'cluster': row['cluster'], 'n_obs': n, 'missions': missions})
    time.sleep(0.3)

for r in results:
    print(f"{r['cluster']:15s} {r['GaiaDR3_ID']}  n={r['n_obs']}  {r['missions']}")

APer            251387120195665152  n=1  ['TESS']
Collinder135    5589411172768195840  n=16  ['TESS']
HPer            458360490378051456  n=25  ['TESS']
Hyades          144506309274714624  n=28  ['TESS', 'K2']
IC2391          5317887321743547264  n=3  ['TESS']
IC2602          5239799391792479488  n=11  ['TESS']
M34             337180328879557120  n=10  ['TESS']
M35             3425496835312410112  n=0  ['error: Could not resolve "Gaia DR3 3425496835312410112" to a sky position.']
M37             3450988286555378176  n=1  ['TESS']
M50             3050039590390925056  n=3  ['TESS']
M67             598883642685066368  n=274  ['TESS', 'K2']
NGC1647         3409891359404877696  n=70  ['TESS', 'K2']
NGC1750         3418675155938540672  n=147  ['TESS', 'K2']
NGC1758         3418695630046147840  n=156  ['TESS', 'K2']
NGC1817         3393981052490356992  n=227  ['TESS', 'K2']
NGC2281         945468231955682560  n=17  ['TESS']
NGC2451A        5538729734043145600  n=5  ['TESS']
NGC2516         52

In [34]:
obs = Observations.query_criteria(
    objectname="Gaia DR3 3393981052490356992",  # NGC1817 star
    obs_collection=['Kepler', 'K2', 'TESS'],
    dataproduct_type='timeseries'
)

print(obs[['obs_collection', 'obs_id', 'target_name', 't_exptime', 'filters', 'provenance_name']].to_pandas().to_string(index=False))

obs_collection                                               obs_id   target_name  t_exptime filters provenance_name
          TESS tess2020324010417-s0032-0000000411697517-0200-a_fast     411697517       20.0    TESS            SPOC
          TESS      tess2021284114741-s0044-0000000411697517-0215-s     411697517      120.0    TESS            SPOC
          TESS       tess2018206190142-s0001-s0046-0000000411697517     411697517      120.0    TESS            SPOC
          TESS       tess2021233042500-s0042-s0046-0000000411697517     411697517      120.0    TESS            SPOC
          TESS      tess2021258175143-s0043-0000000311365280-0214-s     311365280      120.0    TESS            SPOC
          TESS      tess2021284114741-s0044-0000000311365280-0215-s     311365280      120.0    TESS            SPOC
          TESS       tess2018235142541-s0002-s0072-0000000411691547     411691547      120.0    TESS            SPOC
          TESS      tess2020324010417-s0032-0000000311365280-020

In [35]:
from astroquery.mast import Catalogs

# TIC has a Gaia DR2 crossmatch built in — test on a few unrecovered stars
test_ids = unrecovered.head(5)['GaiaDR3_ID'].tolist()

for dr3_id in test_ids:
    try:
        result_tic = Catalogs.query_criteria(
            catalog='TIC',
            GAIA=dr3_id  # TIC stores Gaia DR2 IDs in the GAIA column
        )
        tic_ids = result_tic['ID'].tolist() if len(result_tic) > 0 else []
        print(f"DR3 {dr3_id} → TIC: {tic_ids}")
    except Exception as e:
        print(f"DR3 {dr3_id} → Error: {e}")

DR3 5290646440127895936 → TIC: ['372912811']
DR3 5290654205428930688 → TIC: ['364398772']
DR3 5290654858263942656 → TIC: ['410452133']
DR3 145203704587705088 → TIC: ['61262664']
DR3 145217379763796992 → TIC: ['118555907']


In [36]:
# Test bulk query — pass a list of Gaia IDs at once
test_bulk = unrecovered.head(20)['GaiaDR3_ID'].tolist()

try:
    result_bulk = Catalogs.query_criteria(
        catalog='TIC',
        GAIA=test_bulk
    )
    print(f"Results: {len(result_bulk)}")
    print(result_bulk['ID', 'GAIA', 'ra', 'dec'][:10])
except Exception as e:
    print(f"Error: {e}")

Results: 21
    ID            GAIA               ra              dec       
--------- ------------------- ---------------- ----------------
118555907  145217379763796992 69.1622035588416 22.9699571918459
150130337  148172179824515968 70.4510512066314 25.5751580851553
184010986 2076389745854474880 295.446474542324 40.1584289588728
184011302 2076388337105190144 295.498263682068  40.083404408941
660230071  144506309274714624 66.9050078833858 20.9439627025897
139109212 2076297867915351040 295.237082938931  40.102791968891
125977598  147441558642852736 71.1130923882213 25.2045666004035
184011049 2076298761268673152 295.377529853186  40.145223112427
139109255 2076298795628345088 295.335445837433 40.1123756458063
139109220 2076297829248112256 295.223864469373 40.1044326708723


In [37]:
# Check TIC documentation column — is GAIA field DR2 or DR3?
# Cross-check one known star where we have both DR2 and DR3
# NGC1817 star we queried earlier: DR3 = 3393981052490356992
test_star = Catalogs.query_criteria(
    catalog='TIC',
    GAIA=[3393981052490356992]
)
print(test_star['ID', 'GAIA', 'ra', 'dec'])

# Also check what version TIC says it uses
print("\nAll columns:")
print(test_star.colnames)

    ID            GAIA              ra             dec      
--------- ------------------- -------------- ---------------
311365456 3393981052490356992 78.21344454922 16.457501847028

All columns:
['ID', 'version', 'HIP', 'TYC', 'UCAC', 'TWOMASS', 'SDSS', 'ALLWISE', 'GAIA', 'APASS', 'KIC', 'objType', 'typeSrc', 'ra', 'dec', 'POSflag', 'pmRA', 'e_pmRA', 'pmDEC', 'e_pmDEC', 'PMflag', 'plx', 'e_plx', 'PARflag', 'gallong', 'gallat', 'eclong', 'eclat', 'Bmag', 'e_Bmag', 'Vmag', 'e_Vmag', 'umag', 'e_umag', 'gmag', 'e_gmag', 'rmag', 'e_rmag', 'imag', 'e_imag', 'zmag', 'e_zmag', 'Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag', 'TWOMflag', 'prox', 'w1mag', 'e_w1mag', 'w2mag', 'e_w2mag', 'w3mag', 'e_w3mag', 'w4mag', 'e_w4mag', 'GAIAmag', 'e_GAIAmag', 'Tmag', 'e_Tmag', 'TESSflag', 'SPFlag', 'Teff', 'e_Teff', 'logg', 'e_logg', 'MH', 'e_MH', 'rad', 'e_rad', 'mass', 'e_mass', 'rho', 'e_rho', 'lumclass', 'lum', 'e_lum', 'd', 'e_d', 'ebv', 'e_ebv', 'numcont', 'contratio', 'disposition', 'duplicat